# Characteristic Function Families on SPY Data

Compare Gaussian, NIG, CGMY, and Lévy-stable fits on SPY returns. We compute AIC and plot CF distance to highlight model quality differences.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from cfad.utils import load_spy_sample
from cfad.models.gaussian import GaussianCF
from cfad.models.nig import NIGCF
from cfad.models.cgmy import CGMYCF
from cfad.models.levy_stable import LevyStableCF
from cfad.empirical_cf import ecf_at

## Load SPY returns

We use daily SPY returns from 2018 to 2022 and compare parametric characteristic function fits.

In [ ]:
returns = load_spy_sample()
returns = returns.loc['2019-01-01':'2021-12-31']
returns.head()

## Fit models and compare AIC

In [ ]:
xi = np.linspace(-15, 15, 256)
ecf = ecf_at(returns.values, xi)
models = [
    GaussianCF(),
    NIGCF(),
    CGMYCF(),
    LevyStableCF(),
]
results = []
for model in models:
    fitted = model.fit(returns.values)
    dist = np.mean(np.abs(ecf - fitted.cf(xi))**2)
    results.append({
        'model': type(fitted).__name__,
        'aic': fitted.aic(returns.values),
        'ecf_l2': dist,
        'repr': repr(fitted),
    })
df = pd.DataFrame(results)
df_sorted = df.sort_values('ecf_l2')
df_sorted

## CF distance comparison

Plot the difference between the empirical characteristic function and each fitted model.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for model in models:
    fit_model = model.fit(returns.values)
    ax.plot(xi, np.abs(ecf - fit_model.cf(xi)), label=type(fit_model).__name__)
ax.set_xlabel(r'i')
ax.set_ylabel('ECF L2 error')
ax.set_title('ECF distance for fitted CF families')
ax.legend()
ax.grid(True, alpha=0.3)
figure_path = Path('paper/figures/02_cf_distance.png')
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=150)
print(f'Saved CF distance figure to {figure_path}')